# 02 · A checkmaite capability on *my* cluster

**Who this is for:** a data scientist who wants to run checkmaite's evaluation capabilities
(`import checkmaite`) against a dataset, with the heavy lifting on the Ray cluster the previous
notebook started — not on the notebook kernel.

checkmaite's job backend talks to Ray directly (`ray://…:10001`, the Ray Client). That port is
reachable **only from your own notebook pod** — the per-owner NetworkPolicy Bifrost writes is the
gate — so nothing here carries a credential: reachability is the permission.

Run `01-my-cluster.ipynb` first.

In [ ]:
# The Bifrost sidebar talks to a small server extension inside this very
# JupyterLab (`/user/<you>/bifrost/*`). A notebook can call the same routes
# with the server's own hub token, so what happens here is exactly what a
# click in the sidebar does: same identity, same project, same NetworkPolicy.
import os, time, json, requests

SERVER = os.environ["JUPYTERHUB_SERVICE_URL"]          # http://0.0.0.0:8888/user/<you>/
USER = os.environ.get("JUPYTERHUB_USER", "me")
_HDR = {"Authorization": f"token {os.environ['JUPYTERHUB_API_TOKEN']}"}

def ext(method, path, body=None, **kw):
    """Call an extension route; returns (status, json-or-text)."""
    r = requests.request(method, SERVER + "bifrost/" + path, headers=_HDR, json=body, timeout=60, **kw)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text

def my_running_cluster():
    """The cluster these notebooks share: BIFROST_CLUSTER_ID if set, else the one running cluster."""
    want = os.environ.get("BIFROST_CLUSTER_ID")
    status, view = ext("GET", "clusters")
    assert status == 200 and view.get("configured", True), f"extension not configured: {status} {view}"
    running = [c for c in view["clusters"] if c["state"] == "running"]
    if want:
        return next((c for c in running if c["id"] == want), None)
    return running[0] if len(running) == 1 else None

def wait_running(cluster_id, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        status, view = ext("GET", "clusters")
        state = next((c["state"] for c in view["clusters"] if c["id"] == cluster_id), "gone")
        print(f"{time.time()-t0:5.0f}s  {cluster_id}: {state}", flush=True)
        if state == "running":
            return
        if state in ("failed", "terminated", "gone"):
            raise RuntimeError(f"cluster {cluster_id} went {state}")
        time.sleep(10)
    raise TimeoutError(f"cluster {cluster_id} not running after {timeout}s")

print("notebook user:", USER, "| server:", SERVER)

In [ ]:
import ray
from bifrost_jupyter._address import ray_client_address
from bifrost_jupyter.config import default_namespace

cluster = my_running_cluster()
assert cluster, "no running cluster: run 01-my-cluster.ipynb first (or set BIFROST_CLUSTER_ID)"
CLUSTER_ID = cluster["id"]
RAY_ADDRESS = ray_client_address(CLUSTER_ID, default_namespace())
print("cluster", CLUSTER_ID, "->", RAY_ADDRESS)

## Versions and the analytics volume

The notebook and the cluster must agree on Python, Ray and checkmaite (the capability code is
shipped by reference). The `checkmaite` profile mounts the analytics volume at
`/app/data/analytics`; that is where the demo datasets are and where results go.

In [ ]:
# What does the cluster have? Same checkmaite/ray/python as this kernel is the
# contract for driving it from here; the analytics volume is the profile's storage.
if ray.is_initialized():
    ray.shutdown()
ray.init(RAY_ADDRESS, logging_level="ERROR")

@ray.remote(num_cpus=0)
def probe(paths):
    import sys, os, ray as _r, checkmaite as _c
    return {
        "python": sys.version.split()[0], "ray": _r.__version__, "checkmaite": _c.__version__,
        "mounted": {p: os.path.isdir(p) for p in paths},
        "node": _r.get_runtime_context().get_node_id()[:8],
    }

import checkmaite, sys
DATA_ROOT = os.environ.get("CHECKMAITE_DATA_ROOT", "/app/data/analytics")
remote = ray.get(probe.remote([DATA_ROOT, f"{DATA_ROOT}/datasets"]))
local = {"python": sys.version.split()[0], "ray": ray.__version__, "checkmaite": checkmaite.__version__}
print("notebook:", local)
print("cluster :", remote)
for k in local:
    assert local[k].split(".")[:2] == remote[k].split(".")[:2], f"{k} differs: {local[k]} vs {remote[k]}"
ANALYTICS_MOUNTED = remote["mounted"][DATA_ROOT]
print("analytics volume on the cluster:", ANALYTICS_MOUNTED)
ray.shutdown()

In [ ]:
# The dataset is opened where the files are. With the analytics volume mounted the
# demo datasets are on every node and the dataset object is built by a task on the
# cluster (its files never come to the notebook). Without it, a small synthetic
# YOLO-classification set is generated here and shipped with the runtime_env, so
# the notebook still works — slower and ephemeral, but it works.
DATASET_NAME = os.environ.get("CHECKMAITE_DATASET", "demo-ic-baseline")
runtime_env = {}

if ANALYTICS_MOUNTED:
    DATA_DIR = f"{DATA_ROOT}/datasets/{DATASET_NAME}"
    STORE_URI = f"{DATA_ROOT}/notebooks/{USER}"

    def build_dataset(root, split, dataset_id=None):
        @ray.remote(num_cpus=0)
        def _build(root, split, dataset_id):
            from checkmaite.core.image_classification.dataset_loaders import YoloClassificationDataset
            ds = YoloClassificationDataset(root, split=split, dataset_id=dataset_id)
            return ds, len(ds)
        return ray.get(_build.remote(root, split, dataset_id))
else:
    import pathlib, random
    from PIL import Image, ImageDraw
    local_root = pathlib.Path("synthetic-ic"); DATA_DIR = "synthetic-ic"
    for label in ("square", "circle"):
        d = local_root / "val" / label; d.mkdir(parents=True, exist_ok=True)
        for i in range(12):
            im = Image.new("RGB", (64, 64), (random.randint(180, 255),) * 3); dr = ImageDraw.Draw(im)
            box = (random.randint(4, 20), random.randint(4, 20), random.randint(40, 60), random.randint(40, 60))
            (dr.rectangle if label == "square" else dr.ellipse)(box, fill=(random.randint(0, 120), 0, random.randint(0, 120)))
            im.save(d / f"{label}_{i:02d}.png")
    runtime_env = {"working_dir": str(local_root.parent.resolve()), "excludes": ["*.ipynb", ".ipynb_checkpoints"]}
    STORE_URI = "/tmp/checkmaite-analytics"   # on the cluster; gone with it
    print("no analytics volume on the cluster -> synthetic dataset shipped via runtime_env")

    def build_dataset(root, split, dataset_id=None):
        from checkmaite.core.image_classification.dataset_loaders import YoloClassificationDataset
        ds = YoloClassificationDataset(root, split=split, dataset_id=dataset_id)
        return ds, len(ds)

print("dataset dir:", DATA_DIR, "| analytics store:", STORE_URI)

## Configure the job backend and submit

`configure_job_backend` is the whole integration: an address, a durable analytics store, and a
scope so re-submitting the same run is deduplicated rather than repeated. `submit_capability`
returns a job handle you can poll, wait on, or cancel — the run itself happens on the cluster.

In [ ]:
from checkmaite.jobs import configure_job_backend, submit_capability
from checkmaite.core.image_classification import DataevalCleaning

configure_job_backend(
    "ray",
    address=RAY_ADDRESS,
    runtime_env=runtime_env or None,
    analytics_store={"backend": "parquet", "uri": STORE_URI},
    idempotency_scope=f"notebook-{USER}",
    force_reinit=True,
)
ds, n = build_dataset(DATA_DIR, "val")
print(f"dataset {ds.metadata['id']}: {n} images, labels {ds.metadata['index2label']}")

t0 = time.time()
job = submit_capability(DataevalCleaning(), datasets=[ds])
print("job", job.job_id, "status", job.status())
status = job.wait(timeout=900)
print(f"finished {status.name} in {time.time()-t0:.0f}s")
ref = job.result()
print("run_uid   :", ref.run_uid)
print("capability:", ref.capability_id)
print("store_uri :", ref.store_uri)

## Read the results back

The analytics store is parquet on the cluster's volume. Reading it from a Ray task keeps the
data where it is; only the rows come to the notebook.

In [ ]:
@ray.remote(num_cpus=0)
def read_rows(uri):
    import polars as pl, glob, os
    files = sorted(glob.glob(os.path.join(uri, "**", "*.parquet"), recursive=True)) if os.path.isdir(uri) else [uri]
    df = pl.concat([pl.read_parquet(f) for f in files], how="diagonal_relaxed") if files else pl.DataFrame()
    return df.to_pandas(), files

rows, files = ray.get(read_rows.remote(ref.store_uri or STORE_URI))
print(len(files), "parquet file(s) under", ref.store_uri or STORE_URI)
rows.head(20)

In [ ]:
if ref.report is not None:
    print(type(ref.report).__name__)
    print(json.dumps(ref.report.model_dump(mode="json"), indent=1, default=str)[:3000])
else:
    print("this capability produced no inline report; the analytics rows above are the result")

## Where this leaves you

Everything above is plain checkmaite; the only Bifrost-specific lines were the two that found the
cluster and its address. Notebook 03 uses the same two lines to put the cluster under real load.